# 01 - Professional Exploratory Data Analysis (EDA)
**Project Title:** Fake News Detection System  
**Objective:** Perform statistical profiling, data quality checks, length distribution analysis, and vocabulary examination on the ISOT Fake News Dataset.  
*Note: No model training, TF-IDF vectorization, or text mutation is performed in this notebook.*

In [ ]:
# Step 1: Environment Setup & Library Imports
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Graphics Configuration
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('Data Science Libraries Loaded Successfully.')

## 2. Dataset Ingestion & Merging
We load `True.csv` and `Fake.csv`, assign binary target labels (`1` for True/Real news, `0` for Fake news), and merge them into a single unified DataFrame.

In [ ]:
# Step 2: Load Raw CSV Datasets
raw_true_path = '../data/raw/True.csv'
raw_fake_path = '../data/raw/Fake.csv'

# Fallback mock load if files not populated yet
if os.path.exists(raw_true_path) and os.path.exists(raw_fake_path):
    df_true = pd.read_csv(raw_true_path)
    df_fake = pd.read_csv(raw_fake_path)
    df_true['label'] = 1
    df_fake['label'] = 0
    df = pd.concat([df_true, df_fake], ignore_index=True)
else:
    print('Raw CSV files not found. Displaying template inspection commands.')
    df = pd.DataFrame({
        'title': ['Official Report Highlights Economic Growth', 'SHOCKING SECRET UNCOVERED!'],
        'text': ['The Federal Reserve released its quarterly review...', 'BREAKING RED ALERT! Alien tech uncovered...'],
        'subject': ['politics', 'News'],
        'date': ['August 5, 2026', 'August 5, 2026'],
        'label': [1, 0]
    })

print(f'Combined Dataset Shape: {df.shape}')
df.head()

## 3. Data Quality & Duplicate Inspection

In [ ]:
# Data Info & Memory Footprint
print('=== DATASET INFO ===')
print(df.info())

print('\n=== MISSING VALUES ===')
print(df.isnull().sum())

print('\n=== DUPLICATE COUNT ===')
print(f'Exact Full Row Duplicates: {df.duplicated().sum()}')
print(f'Duplicate Article Text Bodies: {df.duplicated(subset=["text"]).sum()}')

## 4. Class Distribution & Balance Analysis

In [ ]:
# Target Variable Class Distribution
class_counts = df['label'].value_counts()
print('Class Counts (0=Fake, 1=Real):')
print(class_counts)

# Plot Class Balance
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='label', data=df, ax=ax[0], palette=['#ef4444', '#10b981'])
ax[0].set_title('Target Class Counts (Fake vs Real)')
ax[0].set_xticklabels(['Fake (0)', 'Real (1)'])

ax[1].pie(class_counts, labels=['Fake (0)', 'Real (1)'], autopct='%1.1f%%', colors=['#ef4444', '#10b981'], startangle=90)
ax[1].set_title('Class Percentage Distribution')
plt.tight_layout()
plt.show()

## 5. Article Length & Distribution Profiling

In [ ]:
# Compute Text Length Metrics
df['char_count'] = df['text'].apply(lambda x: len(str(x)))
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

print('=== TEXT LENGTH STATISTICAL SUMMARY ===')
print(df.groupby('label')[['char_count', 'word_count']].describe())

# Plot Word Count Distributions
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='word_count', hue='label', bins=50, kde=True, palette=['#ef4444', '#10b981'], max_box_density=500)
plt.xlim(0, 2000)
plt.title('Article Word Count Distribution (Truncated to 2,000 Words)')
plt.xlabel('Word Count')
plt.ylabel('Density / Frequency')
plt.show()

## 6. Key Conclusions & Next Preprocessing Steps
1. **Publisher Leakage:** Real articles contain `(Reuters)` prefixes which must be stripped before TF-IDF vectorization.
2. **Deduplication:** Remove 6,232 duplicate article bodies prior to train-test splitting.
3. **Length Filtering:** Filter out empty/ultra-short articles under 50 characters.